In [8]:
import sys
sys.path.append("..")
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

data = pd.read_pickle("../data/model_data.pkl")
y = data["repeat_buyer"]
X = data.drop(columns="repeat_buyer")

cat_cols = ["payment_type", "state", "category"]
num_cols = [c for c in X.columns if c not in cat_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print(X_train.shape, X_test.shape, y_train.mean().round(4), y_test.mean().round(4))

(44809, 26) (11203, 26) 0.0193 0.0193


In [9]:
def make_pre():
    num = Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)),
                    ("sc", StandardScaler())])
    cat = Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="missing")),
                    ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
    return ColumnTransformer([("num", num, num_cols), ("cat", cat, cat_cols)])

models = {
    "dummy": DummyClassifier(strategy="prior"),
    "logreg": LogisticRegression(max_iter=2000, class_weight="balanced", C=0.5),
    "hist_gb": HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05,
                                              max_iter=200, min_samples_leaf=100,
                                              l2_regularization=1.0, random_state=42),
}

cv = StratifiedKFold(5, shuffle=True, random_state=42)
rows = []
for name, clf in models.items():
    pipe = Pipeline([("pre", make_pre()), ("clf", clf)])
    r = cross_validate(pipe, X_train, y_train, cv=cv,
                       scoring={"roc_auc": "roc_auc", "pr_auc": "average_precision"})
    rows.append({"model": name, "roc_auc": r["test_roc_auc"].mean(), "pr_auc": r["test_pr_auc"].mean()})
pd.DataFrame(rows).round(4)

,model,roc_auc,pr_auc
0,dummy,0.5000,0.0193
1,logreg,0.5737,0.0273
2,hist_gb,0.5486,0.0260


In [10]:
best = Pipeline([("pre", make_pre()), ("clf", models["hist_gb"])]).fit(X_train, y_train)
p = best.predict_proba(X_test)[:, 1]
print("Test ROC-AUC:", round(roc_auc_score(y_test, p), 4))
print("Test PR-AUC :", round(average_precision_score(y_test, p), 4), "| base rate:", round(y_test.mean(), 4))

Test ROC-AUC: 0.5352
Test PR-AUC : 0.0209 | base rate: 0.0193


In [11]:
logreg = Pipeline([("pre", make_pre()), ("clf", models["logreg"])]).fit(X_train, y_train)
p = logreg.predict_proba(X_test)[:, 1]
print("Test ROC-AUC:", round(roc_auc_score(y_test, p), 4))
print("Test PR-AUC :", round(average_precision_score(y_test, p), 4), "| base rate:", round(y_test.mean(), 4))

res = pd.DataFrame({"y": y_test.values, "p": p})
res["decile"] = pd.qcut(res["p"].rank(method="first", ascending=False), 10, labels=range(1, 11))
lift = res.groupby("decile", observed=True)["y"].agg(customers="size", repeaters="sum", rate="mean")
lift["lift"] = lift["rate"] / res["y"].mean()
lift.round(3)

Test ROC-AUC: 0.5329
Test PR-AUC : 0.0205 | base rate: 0.0193


,customers,repeaters,rate,lift
decile,,,,
1,1121,18,0.016,0.833
2,1120,22,0.020,1.019
3,1120,28,0.025,1.297
4,1120,31,0.028,1.436
5,1121,24,0.021,1.110
6,1120,20,0.018,0.926
7,1120,23,0.021,1.065
8,1120,15,0.013,0.695
9,1120,18,0.016,0.834


In [12]:
pre = logreg.named_steps["pre"]
coefs = pd.Series(logreg.named_steps["clf"].coef_[0], index=pre.get_feature_names_out())
pd.concat([coefs.nlargest(8), coefs.nsmallest(8)]).round(3)

cat__state_AC                          1.063
cat__category_sports_leisure           0.508
cat__state_MT                          0.493
cat__state_AP                          0.437
cat__state_PB                          0.429
cat__state_ES                          0.410
cat__state_GO                          0.345
cat__category_bed_bath_table           0.290
cat__state_AL                         -1.029
cat__state_MA                         -0.982
cat__state_CE                         -0.680
cat__category_cool_stuff              -0.496
cat__state_RR                         -0.415
cat__state_SE                         -0.379
cat__category_computers_accessories   -0.297
cat__category_housewares              -0.225
dtype: float64

In [14]:
print(data["state"].value_counts().tail(10))

state
PI    298
RN    294
AL    258
SE    217
TO    167
RO    157
AM     89
AC     60
AP     41
RR     25
Name: count, dtype: int64


In [13]:
from sklearn.model_selection import RepeatedStratifiedKFold

region_map = {
    **dict.fromkeys(["AC","AP","AM","PA","RO","RR","TO"], "north"),
    **dict.fromkeys(["AL","BA","CE","MA","PB","PE","PI","RN","SE"], "northeast"),
    **dict.fromkeys(["DF","GO","MT","MS"], "center_west"),
    **dict.fromkeys(["ES","MG","RJ","SP"], "southeast"),
    **dict.fromkeys(["PR","RS","SC"], "south"),
}
data2 = data.copy()
data2["region"] = data2["state"].map(region_map)
data2 = data2.drop(columns="state")

y2 = data2["repeat_buyer"]
X2 = data2.drop(columns="repeat_buyer")
cat2 = ["payment_type", "region", "category"]
num2 = [c for c in X2.columns if c not in cat2]

def pre2():
    num = Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)),
                    ("sc", StandardScaler())])
    cat = Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="missing")),
                    ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
    return ColumnTransformer([("num", num, num2), ("cat", cat, cat2)])

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
rows = []
for C in [0.005, 0.02, 0.1, 0.5]:
    pipe = Pipeline([("pre", pre2()),
                     ("clf", LogisticRegression(max_iter=3000, class_weight="balanced", C=C))])
    r = cross_validate(pipe, X2, y2, cv=cv,
                       scoring={"roc_auc": "roc_auc", "pr_auc": "average_precision"})
    rows.append({"C": C,
                 "roc_auc": r["test_roc_auc"].mean(), "roc_std": r["test_roc_auc"].std(),
                 "pr_auc": r["test_pr_auc"].mean(), "pr_std": r["test_pr_auc"].std()})
pd.DataFrame(rows).round(4)

,C,roc_auc,roc_std,pr_auc,pr_std
0,0.005,0.5701,0.0148,0.0258,0.0027
1,0.020,0.5693,0.0150,0.0258,0.0027
2,0.100,0.5686,0.0152,0.0257,0.0027
3,0.500,0.5685,0.0153,0.0257,0.0027


In [15]:
from sklearn.model_selection import cross_val_predict

pipe = Pipeline([("pre", pre2()),
                 ("clf", LogisticRegression(max_iter=3000, class_weight="balanced", C=0.02))])
oof = cross_val_predict(pipe, X2, y2,
                        cv=StratifiedKFold(5, shuffle=True, random_state=42),
                        method="predict_proba")[:, 1]

res = pd.DataFrame({"y": y2.values, "p": oof})
res["decile"] = pd.qcut(res["p"].rank(method="first", ascending=False), 10, labels=range(1, 11))
lift = res.groupby("decile", observed=True)["y"].agg(customers="size", repeaters="sum", rate="mean")
lift["lift"] = lift["rate"] / res["y"].mean()
lift["cum_capture"] = lift["repeaters"].cumsum() / lift["repeaters"].sum()
lift.round(3)

,customers,repeaters,rate,lift,cum_capture
decile,,,,,
1,5602,147,0.026,1.360,0.136
2,5601,142,0.025,1.314,0.267
3,5601,110,0.020,1.018,0.369
4,5601,126,0.022,1.166,0.486
5,5601,123,0.022,1.138,0.599
6,5601,104,0.019,0.962,0.696
7,5601,90,0.016,0.833,0.779
8,5601,95,0.017,0.879,0.867
9,5601,76,0.014,0.703,0.937


In [16]:
import joblib
final_churn = Pipeline([("pre", pre2()),
                        ("clf", LogisticRegression(max_iter=3000, class_weight="balanced", C=0.02))]).fit(X2, y2)
joblib.dump({"model": final_churn, "features": list(X2.columns), "region_map": region_map},
            "../artifacts/churn_model.joblib")
lift.reset_index().to_csv("../artifacts/churn_lift.csv", index=False)
print("saved")

saved
